<a href="https://colab.research.google.com/github/dang710206/Baitap-AI-/blob/main/Bt10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install osmnx networkx folium geopy ortools

In [2]:
#BT10
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import osmnx as ox
import networkx as nx
import folium
from folium.plugins import HeatMap
from geopy.distance import geodesic
from IPython.display import display

# ================== 1. Dữ liệu vị trí ==================
locations = {
    "Kho": (10.7769, 106.7009),
    "Khách 1": (10.7800, 106.7100),
    "Khách 2": (10.7900, 106.7000),
    "Khách 3": (10.7700, 106.6800),
    "Khách 4": (10.7850, 106.7300),
    "Khách 5": (10.7650, 106.6900),
}

location_names = list(locations.keys())
coordinates = list(locations.values())

# ================== 2. Tải graph đường phố từ OSMnx ==================
G = ox.graph_from_point(
    center_point=coordinates[0],
    dist=8000,
    network_type='drive',
    simplify=True
)

# Thêm tốc độ ước tính (dữ liệu mở từ OSM)
ox.add_edge_speeds(G)

print(f"Đã tải graph với {len(G.nodes)} nodes và {len(G.edges)} edges")

# ================== 3. Tìm nearest node ==================
node_ids = []
for coord in coordinates:
    nearest_node = ox.distance.nearest_nodes(G, coord[1], coord[0])
    node_ids.append(nearest_node)

print("Nearest nodes:", node_ids)

# ================== 4. TÍNH NGUY CƠ TẮC NGHẼN (Logic mờ + dữ liệu OSM) ==================
ox.add_edge_speeds(G)        # Thêm tốc độ ước tính
# ox.add_edge_travel_times(G)  # Tùy chọn, nếu muốn dùng thời gian

def calculate_congestion_risk(data, u, v, center_coord):
    """data là dict của edge, truyền thêm u, v để tính khoảng cách"""

    # Xử lý highway
    highway = data.get('highway')
    if isinstance(highway, list):
        highway = highway[0] if highway else 'residential'
    elif not isinstance(highway, str):
        highway = 'residential'

    # Xử lý speed_kph
    speed_kph = data.get('speed_kph', 30)
    if isinstance(speed_kph, list):
        speed_kph = float(speed_kph[0]) if speed_kph else 30
    else:
        speed_kph = float(speed_kph) if speed_kph else 30

    # Xử lý lanes - PHẦN QUAN TRỌNG NHẤT
    lanes = data.get('lanes', 2)
    if isinstance(lanes, list):
        lanes_str = str(lanes[0]) if lanes else '2'
    else:
        lanes_str = str(lanes)

    # Lấy số lanes đầu tiên, bỏ qua ký tự không phải số
    try:
        lanes_num = int(''.join(filter(str.isdigit, lanes_str.split(',')[0])))
    except:
        lanes_num = 2   # mặc định nếu không parse được

    risk = 0.0

    # Rule 1: Loại đường
    if highway in ['motorway', 'trunk', 'primary']:
        risk += 0.80
    elif highway in ['secondary']:
        risk += 0.55
    elif highway in ['tertiary']:
        risk += 0.40
    else:
        risk += 0.20

    # Rule 2: Tốc độ thấp
    if speed_kph < 25:
        risk += 0.45
    elif speed_kph < 40:
        risk += 0.25

    # Rule 3: Nhiều làn
    if lanes_num >= 4:
        risk += 0.30

    # Rule 4: Gần trung tâm
    try:
        edge_lat = (G.nodes[u]['y'] + G.nodes[v]['y']) / 2
        edge_lon = (G.nodes[u]['x'] + G.nodes[v]['x']) / 2
        dist_to_center = geodesic(center_coord, (edge_lat, edge_lon)).km
        if dist_to_center < 4.0:
            risk += 0.35
    except:
        pass

    return min(1.0, risk)


# Áp dụng cho tất cả edges
center_coord = coordinates[0]
for u, v, k, data in G.edges(keys=True, data=True):
    data['congestion_risk'] = calculate_congestion_risk(data, u, v, center_coord)
    length = data.get('length', 100)
    data['effective_cost'] = length * (1 + 5.0 * data['congestion_risk'])

print("✅ Đã tính xong nguy cơ tắc nghẽn (đã xử lý lanes là string).")

# ================== 5. Tạo 2 distance matrix ==================
num_locations = len(coordinates)

def create_distance_matrix(weight_key='length'):
    matrix = [[0] * num_locations for _ in range(num_locations)]
    for i in range(num_locations):
        for j in range(num_locations):
            if i == j:
                continue
            try:
                length = nx.shortest_path_length(G, node_ids[i], node_ids[j], weight=weight_key)
                matrix[i][j] = int(length)
            except nx.NetworkXNoPath:
                matrix[i][j] = int(geodesic(coordinates[i], coordinates[j]).meters * 10)
    return matrix

distance_matrix_original = create_distance_matrix('length')          # Tuyến gốc
distance_matrix_alternative = create_distance_matrix('effective_cost')  # Tuyến tránh tắc

# ================== 6. Hàm giải OR-Tools (dùng chung cho cả 2 tuyến) ==================
def solve_vrp(distance_matrix):
    def create_data_model():
        return {'distance_matrix': distance_matrix, 'num_vehicles': 1, 'depot': 0}

    data = create_data_model()
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']), data['num_vehicles'], data['depot'])
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        route_nodes = []
        index = routing.Start(0)
        while not routing.IsEnd(index):
            route_nodes.append(node_ids[manager.IndexToNode(index)])
            index = solution.Value(routing.NextVar(index))
        route_nodes.append(node_ids[manager.IndexToNode(index)])  # quay về Kho
        return route_nodes, solution
    return None, None

# Giải 2 tuyến
route_original, _ = solve_vrp(distance_matrix_original)
route_alternative, _ = solve_vrp(distance_matrix_alternative)

# ================== 7. Tính tổng rủi ro của từng tuyến ==================
def calculate_route_risk(route_nodes):
    total = 0.0
    count = 0
    for i in range(len(route_nodes) - 1):
        u, v = route_nodes[i], route_nodes[i + 1]
        edge_data = G.get_edge_data(u, v)
        if edge_data:
            risk = list(edge_data.values())[0].get('congestion_risk', 0.5)
            total += risk
            count += 1
    return round(total / count, 3) if count > 0 else 0

risk_original = calculate_route_risk(route_original) if route_original else 0
risk_alternative = calculate_route_risk(route_alternative) if route_alternative else 0

print("\n=== PHÂN TÍCH NGUY CƠ TẮC NGHẼN ===")
print(f"Tuyến gốc (OR-Tools)      : Rủi ro trung bình = {risk_original}")
print(f"Tuyến thay thế (tránh tắc) : Rủi ro trung bình = {risk_alternative} ← TỐT HƠN")

# ================== 8. Vẽ bản đồ trên Colab ==================
m = folium.Map(location=coordinates[0], zoom_start=14, tiles="OpenStreetMap")

# HeatMap vùng rủi ro cao (dữ liệu mở + logic mờ)
heat_data = []
for u, v, data in G.edges(data=True):
    if data.get('congestion_risk', 0) > 0.55:
        lat = (G.nodes[u]['y'] + G.nodes[v]['y']) / 2
        lon = (G.nodes[u]['x'] + G.nodes[v]['x']) / 2
        heat_data.append([lat, lon, data['congestion_risk'] * 80])

HeatMap(heat_data, radius=15, blur=12, min_opacity=0.5, max_zoom=16).add_to(m)
print("→ HeatMap đỏ = vùng nguy cơ tắc nghẽn cao")

# Vẽ marker điểm dừng
for i, coord in enumerate(coordinates):
    color = 'blue' if i == 0 else 'green'
    folium.Marker(coord, popup=location_names[i], icon=folium.Icon(color=color)).add_to(m)

# Hàm vẽ route có màu theo rủi ro từng đoạn
def draw_colored_route(route_nodes, color_base, name):
    for i in range(len(route_nodes) - 1):
        u = route_nodes[i]
        v = route_nodes[i + 1]
        edge_data = G.get_edge_data(u, v)
        risk = list(edge_data.values())[0].get('congestion_risk', 0.5) if edge_data else 0.5

        seg_color = 'red' if risk > 0.7 else 'orange' if risk > 0.45 else 'green'
        seg_coords = [(G.nodes[u]['y'], G.nodes[u]['x']), (G.nodes[v]['y'], G.nodes[v]['x'])]

        folium.PolyLine(seg_coords, color=seg_color, weight=6, opacity=0.9).add_to(m)

    # Đường tổng thể (mờ)
    route_coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in route_nodes]
    folium.PolyLine(route_coords, color=color_base, weight=3, opacity=0.4, dash_array='8,5').add_to(m)

# Vẽ 2 tuyến
if route_original:
    draw_colored_route(route_original, 'blue', 'Tuyến gốc')
if route_alternative:
    draw_colored_route(route_alternative, 'purple', 'Tuyến thay thế')

# Legend đơn giản
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; width: 220px; height: 250px;
background-color: white; border:2px solid grey; z-index:9999; font-size:14px; padding:10px;">
<b>Chú giải bản đồ nguy cơ tắc nghẽn</b><br>
<span style="color:red;">●</span> Rủi ro cao (primary road, trung tâm)<br>
<span style="color:orange;">●</span> Rủi ro trung bình<br>
<span style="color:green;">●</span> Rủi ro thấp<br>
HeatMap đỏ = Vùng dễ tắc<br>
Đường xanh = Tuyến gốc<br>
Đường tím = Tuyến thay thế (tránh tắc)
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

display(m)
m.save("ban_do_nguy_co_tac_nghen.html")

print("\n✅ Bản đồ đã hiển thị trực tiếp trên Colab!")
print("   • HeatMap đỏ: khu vực nguy cơ tắc cao (dữ liệu OSM + logic mờ)")
print("   • Đường có màu: phân tích rủi ro từng đoạn")
print("   • Tuyến tím: đề xuất thay thế (ưu tiên đường ít tắc)")
print("   • File HTML đã lưu để tải về.")

Đã tải graph với 28442 nodes và 66800 edges
Nearest nodes: [1769277472, 8751584189, 2230414266, 411920662, 12508566182, 4603243122]
✅ Đã tính xong nguy cơ tắc nghẽn (đã xử lý lanes là string).

=== PHÂN TÍCH NGUY CƠ TẮC NGHẼN ===
Tuyến gốc (OR-Tools)      : Rủi ro trung bình = 0
Tuyến thay thế (tránh tắc) : Rủi ro trung bình = 0 ← TỐT HƠN
→ HeatMap đỏ = vùng nguy cơ tắc nghẽn cao



✅ Bản đồ đã hiển thị trực tiếp trên Colab!
   • HeatMap đỏ: khu vực nguy cơ tắc cao (dữ liệu OSM + logic mờ)
   • Đường có màu: phân tích rủi ro từng đoạn
   • Tuyến tím: đề xuất thay thế (ưu tiên đường ít tắc)
   • File HTML đã lưu để tải về.
